# Run ABG-HKG-AOG v7
Edit the paths, then run each Python cell.


In [ ]:
from pathlib import Path
import subprocess, os
REPO_DIR=Path('/home/dfli/instance_slot_aog/clean_v18_v39_v42')
TRAIN_CACHE=REPO_DIR/'artifacts/strict_aog_v6/train_strict_aog_terminals.pt'
VAL_CACHE=REPO_DIR/'artifacts/strict_aog_v6/val_strict_aog_terminals.pt'
BUNDLE=REPO_DIR/'artifacts/abg_hkg_aog_v7/abg_hkg_aog_v7_bundle.pt'
PART_TEMPLATE_BANK=REPO_DIR/'artifacts/abg_hkg_aog_v7/part_template_bank.pt'
RUN_DIR=REPO_DIR/'runs/abg_hkg_aog_v7'
BEST_CKPT=RUN_DIR/'checkpoints/strict_aog_best.pt'
def run(cmd):
    print(' '.join(map(str, cmd)))
    subprocess.run([str(x) for x in cmd], cwd=REPO_DIR, check=True)
print(REPO_DIR)

In [ ]:
run(['git','fetch','origin','pra-aog-v6-gpu-terminal-cache'])
run(['git','switch','pra-aog-v6-gpu-terminal-cache'])
run(['git','pull','--ff-only','origin','pra-aog-v6-gpu-terminal-cache'])
run(['python','-m','pip','install','-e','.[dev,vision]'])

In [ ]:
run(['pytest','-q','tests/test_abg_hkg_aog_v7.py','tests/test_pra_aog_v6_template_hierarchy.py','tests/test_hier_pra_aog.py'])

In [ ]:
BUNDLE.parent.mkdir(parents=True, exist_ok=True)
run(['python','scripts/build_abg_hkg_aog_v7.py','--cache',TRAIN_CACHE,'--out',BUNDLE,'--part-template-out',PART_TEMPLATE_BANK,'--num-templates-per-class','5','--max-slots-per-template','14','--max-slots-per-part','4','--part-template-grid-size','3','--part-template-min-support','6','--part-template-max-per-part','6'])

In [ ]:
run(['python','scripts/run_abg_hkg_aog_v7.py','--bundle',BUNDLE,'--part-template-bank',PART_TEMPLATE_BANK,'--train-cache',TRAIN_CACHE,'--val-cache',VAL_CACHE,'--save-dir',str(RUN_DIR)+'_smoke','--device','auto','--batch-size','4','--epochs','1','--v7-max-rounds','1','--v7-max-queries','2','--max-train-batches','2','--max-val-batches','2','--num-workers','0'])

In [ ]:
run(['python','scripts/run_abg_hkg_aog_v7.py','--bundle',BUNDLE,'--part-template-bank',PART_TEMPLATE_BANK,'--train-cache',TRAIN_CACHE,'--val-cache',VAL_CACHE,'--save-dir',RUN_DIR,'--device','auto','--batch-size','16','--epochs','20','--v7-max-rounds','1','--v7-max-queries','2','--preload-cache','--num-workers','0'])

In [ ]:
run(['python','scripts/infer_abg_hkg_aog_v7.py','--bundle',BUNDLE,'--part-template-bank',PART_TEMPLATE_BANK,'--cache',VAL_CACHE,'--checkpoint',BEST_CKPT,'--out-dir',RUN_DIR/'inference','--sample-index','0','--device','auto','--v7-max-rounds','1','--v7-max-queries','2'])